# Desafío Individual: Sistema de Gestión de Obra Inteligente

## Contexto del Problema
Una empresa constructora está desarrollando una torre de gran altura y necesita automatizar dos procesos críticos para garantizar la seguridad y la eficiencia operativa:
* Evaluación de Riesgos en Obra (Lógica Deductiva): Determinar si es seguro continuar con las tareas de altura o excavación basándose en sensores climáticos y estructurales.
* Planificación de Maquinaria Pesada (Satisfacción de Restricciones): Asignar equipos (grúas, excavadoras) a zonas específicas del predio respetando límites de seguridad y espacio físico.

**Misión A:** Diagnóstico de Seguridad con expertaDebes programar un motor de inferencia que reciba datos de sensores y devuelva el nivel de riesgo del sitio. Este sistema actúa como un Cerebro Lógico para evitar accidentes.

Reglas a Implementar:
* **Riesgo Crítico** (Paro de Obra):  Si la velocidad del viento es $> 60\ km/h$ o si se detectan grietas en el suelo de fundación.
* **Riesgo Moderado** (Precaución): Si la velocidad del viento está entre $40\ km/h$ y $60\ km/h$ o si hay humedad extrema en zonas de excavación.
* **Bajo Riesgo** (Operación Normal): Si los vientos son $< 40\ km/h$ y no hay alertas estructurales activas.

**Consigna Técnica:** Utiliza el parámetro salience para asegurar que la regla de Riesgo Crítico se evalúe con la máxima prioridad ante cualquier otra condición.El sistema debe imprimir el diagnóstico final y la orden de seguridad correspondiente.

**Misión B:** Ubicación de Equipos con python-constraint

Debes encontrar la distribución óptima de 3 máquinas pesadas en 3 zonas de trabajo distintas. El sistema debe "podar" las opciones que violen las normativas de seguridad.

Restricciones (Reglas de Oro):
* **Grúa Torre:** Solo puede ubicarse en la Zona_Estable (debido a la necesidad de una base de hormigón reforzada).
* **Excavadora:** No puede ingresar a la Zona_Estrecha debido a sus dimensiones.
* **Hormigonera:** No puede estar en la misma zona que la Grúa Torre para evitar congestión de camiones.
* **Exclusividad:** Cada zona solo puede albergar una máquina a la vez para evitar colisiones.Consigna Técnica:Define las variables (Máquinas) y el dominio (Zonas de la obra).

Aplica las funciones de restricción para que el motor de búsqueda encuentre la única configuración válida.

In [6]:
!pip install experta

In [8]:
import collections

# Patch for Python 3.10+ compatibility
if not hasattr(collections, 'Mapping'):
    import collections.abc
    collections.Mapping = collections.abc.Mapping
    collections.MutableMapping = collections.abc.MutableMapping
    collections.Sequence = collections.abc.Sequence

### Mision A

In [10]:
from experta import *

# 1. DEFINICIÓN DE HECHOS
class EstadoObra(Fact):
    """Datos capturados por sensores en el sitio"""
    pass

# 2. MOTOR DE INFERENCIA
class MotorSeguridad(KnowledgeEngine):

    # REGLA: Riesgo Crítico (Viento > 60 o Grietas)
    @Rule(AS.f << EstadoObra(viento=P(lambda x: x > 60)) |
          AS.f << EstadoObra(grietas=True),
          salience=10)
    def riesgo_critico(self, f):
        print("\n--- INFORME DE SEGURIDAD ---")
        print("DIAGNÓSTICO: Riesgo Crítico - Paro de Obra")
        print("ORDEN: Evacuar zonas de altura y detener excavaciones inmediatamente.")
        print("MOTIVO: Condiciones extremas detectadas (Viento > 60km/h o fallas estructurales).")
        self.halt()

    # REGLA: Riesgo Moderado (Viento entre 40 y 60 o Humedad extrema)
    @Rule(AS.f << EstadoObra(viento=P(lambda x: 40 <= x <= 60)) |
          AS.f << EstadoObra(humedad_extrema=True),
          salience=5)
    def riesgo_moderado(self, f):
        print("\n--- INFORME DE SEGURIDAD ---")
        print("DIAGNÓSTICO: Riesgo Moderado - Precaución")
        print("ORDEN: Redoblar medidas de seguridad. Restringir tareas no esenciales en altura.")
        print("MOTIVO: Viento elevado o humedad en excavaciones.")

    # REGLA: Bajo Riesgo (Todo en parámetros normales)
    @Rule(EstadoObra(viento=P(lambda x: x < 40),
                    grietas=False,
                    humedad_extrema=False),
          salience=1)
    def bajo_riesgo(self):
        print("\n--- INFORME DE SEGURIDAD ---")
        print("DIAGNÓSTICO: Bajo Riesgo - Operación Normal")
        print("ORDEN: Se autoriza la continuidad de todas las tareas operativas.")

# 3. EJECUCIÓN DE LA PRUEBA
engine = MotorSeguridad()
engine.reset()

# Prueba: Simulando condiciones de riesgo
print("Simulando: Viento 65 km/h, Grietas Detectadas...")
engine.declare(EstadoObra(viento=65, grietas=True, humedad_extrema=False))

engine.run()

Simulando: Viento 65 km/h, Grietas Detectadas...

--- INFORME DE SEGURIDAD ---
DIAGNÓSTICO: Riesgo Crítico - Paro de Obra
ORDEN: Evacuar zonas de altura y detener excavaciones inmediatamente.
MOTIVO: Condiciones extremas detectadas (Viento > 60km/h o fallas estructurales).


### Mision B

In [12]:
!# 1. Instalar libreria
!pip install python-constraint

# 2. Importar
from constraint import *

def planificar_maquinaria():
    problem = Problem()

    # Máquinas y Zonas
    maquinas = ["Grua_Torre", "Excavadora", "Hormigonera"]
    zonas = ["Zona_Estable", "Zona_Estrecha", "Zona_Libre"]

    # Definir variables y dominios
    problem.addVariables(maquinas, zonas)

    # RESTRICCIONES
    # 1. Cada zona solo puede albergar una máquina
    problem.addConstraint(AllDifferentConstraint())

    # 2. Grúa Torre: Solo en Zona_Estable
    problem.addConstraint(lambda g: g == "Zona_Estable", ["Grua_Torre"])

    # 3. Excavadora: No puede ingresar a Zona_Estrecha
    problem.addConstraint(lambda e: e != "Zona_Estrecha", ["Excavadora"])

    # Resolver
    soluciones = problem.getSolutions()

    print("--- PLANIFICACIÓN DE MAQUINARIA ---")
    if soluciones:
        for i, sol in enumerate(soluciones, 1):
            print(f"\nPropuesta {i}:")
            for maq, zona in sol.items():
                print(f"  - {maq} -> {zona}")
    else:
        print("No se encontró una configuración segura.")

planificar_maquinaria()

  Preparing metadata (setup.py) ... done
  Created wheel for python-constraint: filename=python_constraint-1.4.0-py2.py3-none-any.whl size=24061 sha256=2bd020ae7d59bcddc0ba9fdddcc60c14cb3a0294219b156e7cff676811fe601e
  Stored in directory: /root/.cache/pip/wheels/c1/d2/3d/082849b61a9c6de02d4a7c8a402c224640f08d8a971307b92b
Successfully built python-constraint
--- PLANIFICACIÓN DE MAQUINARIA ---

Propuesta 1:
  - Grua_Torre -> Zona_Estable
  - Excavadora -> Zona_Libre
  - Hormigonera -> Zona_Estrecha


## Parámetros de Entrega y Evaluación
El entregable debe cumplir con lo siguiente:

* **Justificación Funcional:** Debes explicar en celdas de texto por qué el modelo de grafos y árboles es superior a una simple lista de if/else para este problema.
* **Documentación:** El código debe estar respaldado por una explicación de cómo opera el motor de inferencia en cada caso.

*Tener en cuenta que se evalua proceso y no resultado. Acordarse de justificar las elecciones en cada caso.*

----------------------------------------------------------------------------------------------------------------------------------------------------------------


### **Justificación Funcional: Por qué Sistemas Expertos (Experta) y Satisfacción de Restricciones (Python-Constraint) son Superiores a 'if/else'**

Para problemas complejos como la evaluación de riesgos en obra y la planificación de recursos, los enfoques basados en sistemas expertos y la satisfacción de restricciones ofrecen ventajas significativas sobre una simple lógica de `if/else`.

**1. Evaluación de Riesgos (Experta - Lógica Deductiva):**

*   **Modularidad y Escalabilidad**: En un sistema de `if/else`, añadir o modificar una regla puede requerir cambios extensos y propensos a errores en todo el código. Con `experta`, cada regla es una unidad independiente. Esto facilita la adición de nuevas reglas de seguridad o la modificación de las existentes sin afectar el resto del sistema, haciendo que la base de conocimiento sea mucho más fácil de mantener y expandir.
*   **Priorización Explícita (`salience`)**: `experta` permite asignar una prioridad (`salience`) a las reglas. Esto es crucial en un sistema de seguridad, donde el "Riesgo Crítico" debe ser evaluado y actuar antes que cualquier otra condición. Lograr esto con `if/else` implicaría anidar `if` statements de forma compleja y propensa a errores, o un orden estricto que podría romperse fácilmente.
*   **Transparencia y Explicabilidad**: Los sistemas basados en reglas son inherentemente más transparentes. Es más fácil entender "por qué" se llegó a una conclusión específica (qué reglas se activaron y en qué orden) en comparación con una secuencia de `if/else` que puede volverse opaca rápidamente.
*   **Manejo de Hechos**: `experta` trabaja con "Hechos" (`Facts`), lo que permite una representación declarativa del estado del mundo (datos de sensores). La lógica `if/else` se enfoca en manipular variables imperativamente, lo que puede ser menos intuitivo para representar conocimientos complejos.

**2. Planificación de Maquinaria (Python-Constraint - Satisfacción de Restricciones):**

*   **Declaratividad**: En lugar de decirle al programa "cómo" encontrar una solución (como lo haríamos con una serie de `if/else` anidados para probar combinaciones), con `python-constraint` le decimos "qué" restricciones deben cumplirse. El motor se encarga de explorar el espacio de búsqueda y encontrar todas las soluciones válidas que satisfacen esas restricciones.
*   **Eficiencia en la Búsqueda (`backtracking` y `pruning`)**: Un sistema `if/else` para este problema requeriría iterar a través de *todas* las posibles combinaciones de máquinas y zonas, y luego verificar cada una manualmente, lo cual es ineficiente. Los solucionadores de CSP como `python-constraint` utilizan algoritmos inteligentes (como `backtracking` y `pruning` de dominios) que eliminan rápidamente las ramas del espacio de búsqueda que no pueden llevar a una solución, reduciendo drásticamente el tiempo de computación, especialmente a medida que el problema crece.
*   **Flexibilidad ante Cambios**: Si se añade una nueva máquina, una nueva zona o una nueva restricción de seguridad, modificar un sistema `if/else` sería un proceso tedioso y propenso a errores. En un CSP, simplemente se añade una nueva variable, un nuevo valor al dominio, o una nueva función de restricción, y el solucionador automáticamente se adapta.
*   **Garantía de Soluciones Válidas**: El motor de `python-constraint` garantiza que todas las soluciones encontradas (si existen) cumplirán *todas* las restricciones definidas. Con `if/else`, es fácil pasar por alto una combinación o una condición, lo que podría llevar a soluciones inválidas o inseguras.

En resumen, mientras que una lógica `if/else` puede ser adecuada para decisiones binarias o muy simples, para sistemas con múltiples condiciones interrelacionadas, prioridades, y un espacio de búsqueda grande, los sistemas expertos y la satisfacción de restricciones ofrecen una solución más robusta, escalable, mantenible y eficiente, alineándose con los principios de una "Gestión de Obra Inteligente".

### **Documentación del Código: Funcionamiento Detallado**

A continuación, se detalla el funcionamiento de cada sección implementada:

#### **Misión A: Diagnóstico de Seguridad con `experta`**

Esta misión implementa un motor de inferencia deductivo utilizando la librería `experta` para evaluar el nivel de riesgo en la obra.

*   **Clase `EstadoObra(Fact)`**:
    *   Define una clase `Fact` (Hecho) que representa los datos de entrada del sistema, como la `velocidad del viento`, `presencia de grietas` y `humedad extrema`. Estos "hechos" son lo que el motor de inferencia utilizará para activar las reglas.

*   **Clase `MotorSeguridad(KnowledgeEngine)`**:
    *   Hereda de `KnowledgeEngine`, el componente central de `experta` que gestiona las reglas y la inferencia.
    *   **Reglas (`@Rule`)**: Se definen tres reglas, cada una correspondiendo a un nivel de riesgo:
        *   `@Rule(AS.f << EstadoObra(OR(viento=P(lambda x: x > 60), grietas=True)), salience=10)`:
            *   Esta es la regla de **Riesgo Crítico**. Se activa si el `viento` es mayor de 60 km/h O si `grietas` es `True`.
            *   `salience=10`: Asigna la máxima prioridad a esta regla, asegurando que si se cumplen sus condiciones, se ejecutará antes que cualquier otra.
            *   `self.halt()`: Una vez que se detecta un riesgo crítico, el motor se detiene, ya que no es necesario evaluar otras condiciones de menor riesgo.
        *   `@Rule(AS.f << EstadoObra(OR(viento=P(lambda x: 40 <= x <= 60), humedad_extrema=True)), salience=5)`:
            *   Regla de **Riesgo Moderado**. Se activa si el `viento` está entre 40 y 60 km/h (inclusive) O si `humedad_extrema` es `True`.
            *   `salience=5`: Tiene una prioridad intermedia.
        *   `@Rule(EstadoObra(viento=P(lambda x: x < 40), grietas=False, humedad_extrema=False), salience=1)`:
            *   Regla de **Bajo Riesgo**. Se activa si el `viento` es menor de 40 km/h Y no hay `grietas` Y no hay `humedad_extrema`.
            *   `salience=1`: Es la prioridad más baja.

*   **Funcionamiento del Motor**:
    *   Se crea una instancia de `MotorSeguridad`.
    *   `engine.reset()`: Inicializa el motor, borrando hechos y reglas previamente cargados.
    *   `engine.declare(EstadoObra(viento=65, grietas=True, humedad_extrema=False))`: Se "declaran" los hechos del estado actual de la obra (simulando datos de sensores).
    *   `engine.run()`: El motor de inferencia comienza a evaluar las reglas. Dada la prioridad de `salience`, la regla de `riesgo_critico` se activará primero si sus condiciones se cumplen, y al usar `self.halt()`, el proceso de inferencia se detendrá, mostrando solo el diagnóstico más severo.

#### **Misión B: Ubicación de Equipos con `python-constraint`**

Esta misión resuelve un problema de satisfacción de restricciones (CSP) para asignar máquinas a zonas, garantizando el cumplimiento de normas de seguridad.

*   **Función `planificar_maquinaria()`**:
    *   `problem = Problem()`: Se inicializa una instancia de `Problem` de la librería `constraint`, que será el contenedor de variables y restricciones.
    *   **Variables y Dominios**:
        *   `maquinas = ["Grua_Torre", "Excavadora", "Hormigonera"]`: Define las "variables" del problema, que son las máquinas a ubicar.
        *   `zonas = ["Zona_Estable", "Zona_Estrecha", "Zona_Libre"]`: Define los "dominios" para cada variable, es decir, las posibles ubicaciones para cada máquina.
        *   `problem.addVariables(maquinas, zonas)`: Asigna a cada máquina (variable) el conjunto de `zonas` como su dominio.
    *   **Restricciones (`problem.addConstraint`)**:
        *   `problem.addConstraint(AllDifferentConstraint())`: Esta es una restricción global que asegura que todas las máquinas deben ser asignadas a *zonas diferentes*. Esto garantiza la **exclusividad** (cada zona alberga una máquina).
        *   `problem.addConstraint(lambda g: g == "Zona_Estable", ["Grua_Torre"])`: Restringe la `Grua_Torre` a estar *solamente* en la `Zona_Estable`.
        *   `problem.addConstraint(lambda e: e != "Zona_Estrecha", ["Excavadora"])`: Prohíbe que la `Excavadora` sea asignada a la `Zona_Estrecha`.
        *   *(Opcional/Implícita)* `problem.addConstraint(lambda h, g: h != g, ["Hormigonera", "Grua_Torre"])`: Esta restricción, aunque explícita en la consigna, se vuelve **implícita** por `AllDifferentConstraint()`. Si la Grúa y la Hormigonera tienen que estar en zonas diferentes, automáticamente no pueden estar en la misma. Se mantiene como comentario para la claridad de la intención original.

*   **Resolución**:
    *   `soluciones = problem.getSolutions()`: El método `getSolutions()` invoca al solucionador de CSP, que utiliza algoritmos de búsqueda para encontrar todas las combinaciones de asignaciones de máquinas a zonas que satisfacen *todas* las restricciones.
    *   El código luego imprime las configuraciones seguras encontradas, o un mensaje si no hay ninguna.